<a href="https://colab.research.google.com/github/detektor777/colab_list_image/blob/main/Nanobanana_Watermark_Remover.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title ##**Install** { display-mode: "form" }
%%capture

import subprocess, os, zipfile, shutil

TOOL_PATH   = "/usr/local/bin/GeminiWatermarkTool"
RELEASE_TAG = "v0.2.6"
ASSET_NAME  = "GeminiWatermarkTool-Linux-x64.zip"
DOWNLOAD_URL = (
    f"https://github.com/allenk/GeminiWatermarkTool"
    f"/releases/download/{RELEASE_TAG}/{ASSET_NAME}"
)
ZIP_PATH = "/tmp/GeminiWatermarkTool.zip"

print(f"⬇️  Downloading GeminiWatermarkTool {RELEASE_TAG}...")

# Download the ZIP archive
r = subprocess.run(["wget", "-q", "--show-progress", "-L", "-O", ZIP_PATH, DOWNLOAD_URL])

if r.returncode != 0 or not os.path.exists(ZIP_PATH) or os.path.getsize(ZIP_PATH) < 10_000:
    raise RuntimeError("❌ Download failed. Check your internet connection or the URL.")

print("📦 Extracting...")
with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    # Find the executable file inside the archive (ignore README files or folders)
    files = [f for f in zip_ref.namelist() if not f.endswith('/')]
    bin_filename = next((f for f in files if "GeminiWatermarkTool" in os.path.basename(f)), files[0])
    zip_ref.extract(bin_filename, "/tmp")

# Move the executable to the system directory
extracted_path = os.path.join("/tmp", bin_filename)
shutil.move(extracted_path, TOOL_PATH)

# Make it executable (chmod +x)
os.chmod(TOOL_PATH, os.stat(TOOL_PATH).st_mode | 0o111)

# Check version and functionality
ver = subprocess.run([TOOL_PATH, "--version", "--no-banner"], capture_output=True, text=True)
size_mb = os.path.getsize(TOOL_PATH) / 1024 / 1024

print(f"\n✅ Installed: {(ver.stdout + ver.stderr).strip()}")
print(f"   Size: {size_mb:.1f} MB")
print("\n✅ Done! Proceed to Cell 2 →")

In [ ]:
#@title ##**Upload / Select Files** { display-mode: "form" }
import os
import shutil
import ipywidgets as widgets
from IPython.display import display, clear_output, Image as IPImage
from google.colab import files, drive

upload_option = "Upload from PC" #@param["Upload from PC", "Select Files (Drive)", "Select Directory (Drive)"]

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
ARCHIVE_EXTS = {".zip", ".tar", ".gz"}
WORK_DIR = "/content/processing_input"

INPUT_FILES = []

# ── Palette ──────────────────────────────────────────────────────────────────
C_BG         = "#F0EEE8"   # page surface
C_BORDER     = "#B0ADA4"   # border
C_FOLDER     = "#7A786F"   # folder button bg  (dark enough for white text)
C_FOLDER_TXT = "#FFFFFF"
C_OPEN       = "#5C5A53"   # "Open" button bg
C_OPEN_TXT   = "#FFFFFF"
C_SEL        = "#3D6B4F"   # selected — muted green
C_SEL_TXT    = "#FFFFFF"
C_CONFIRM_BG = "#3A7D44"   # confirm — classic green
C_CONFIRM_TXT= "#FFFFFF"
C_PATH_BG    = "#E4E2DA"
C_PATH_TXT   = "#4A4845"
C_BACK_BG    = "#6B6960"   # back button
C_BACK_TXT   = "#FFFFFF"
# ─────────────────────────────────────────────────────────────────────────────

def clean_work_dir():
    if os.path.exists(WORK_DIR):
        shutil.rmtree(WORK_DIR)
    os.makedirs(WORK_DIR, exist_ok=True)

def is_image(filename):
    return os.path.splitext(filename)[1].lower() in IMAGE_EXTS

def is_archive(filename):
    return os.path.splitext(filename)[1].lower() in ARCHIVE_EXTS

def process_file(filepath):
    if is_image(filepath):
        dest = os.path.join(WORK_DIR, os.path.basename(filepath))
        if filepath != dest:
            shutil.copy(filepath, dest)
        if dest not in INPUT_FILES:
            INPUT_FILES.append(dest)
    elif is_archive(filepath):
        print(f"📦 Extracting: {os.path.basename(filepath)}...")
        extract_dir = os.path.join(WORK_DIR, os.path.basename(filepath) + "_extracted")
        os.makedirs(extract_dir, exist_ok=True)
        try:
            shutil.unpack_archive(filepath, extract_dir)
        except Exception as e:
            print(f"❌ Error extracting {os.path.basename(filepath)}: {e}")
        for root, _, files_in_dir in os.walk(extract_dir):
            for f in files_in_dir:
                if is_image(f):
                    img_path = os.path.join(root, f)
                    if img_path not in INPUT_FILES:
                        INPUT_FILES.append(img_path)

def show_results():
    if not INPUT_FILES:
        print("❌ No images found in selection.")
        return
    print(f"\n✅ Success! Found {len(INPUT_FILES)} images.")
    for img in INPUT_FILES[:3]:
        display(IPImage(img, width=200))
    if len(INPUT_FILES) > 3:
        print(f"...and {len(INPUT_FILES) - 3} more.")
    print("\n✅ Proceed to Cell 3 →")

def _btn_style(bg, txt):
    """Return a Layout + style dict pair for a flat muted button."""
    return dict(
        style=widgets.ButtonStyle(button_color=bg, font_color=txt),
        layout=widgets.Layout(border=f"1px solid {C_BORDER}")
    )

# ══════════════════════════════════════════════════════════════════════════════
# UI LOGIC
# ══════════════════════════════════════════════════════════════════════════════

if upload_option == "Upload from PC":
    uploaded = files.upload()
    if uploaded:
        clean_work_dir()
        INPUT_FILES.clear()
        for filename in uploaded.keys():
            temp_path = os.path.join("/content", filename)
            process_file(temp_path)
            os.remove(temp_path)
        show_results()

else:
    drive.mount('/content/drive')
    ROOT_DRIVE = '/content/drive/MyDrive'

    main_out   = widgets.Output()
    result_out = widgets.Output()

    selected_items = set()

    def draw_browser(current_path):
        with main_out:
            clear_output(wait=True)

            # ── path label ──────────────────────────────────────────────────
            display(widgets.HTML(
                f"<div style='background:{C_PATH_BG}; color:{C_PATH_TXT}; "
                f"padding:6px 10px; border-radius:6px; font-size:12px; "
                f"border:1px solid {C_BORDER}; margin-bottom:6px;'>"
                f"<b>Path:</b> <code style='color:{C_PATH_TXT}'>{current_path}</code></div>"
            ))

            buttons = []

            # ── back button ─────────────────────────────────────────────────
            if current_path != ROOT_DRIVE and current_path.startswith(ROOT_DRIVE):
                btn_up = widgets.Button(
                    description="⬆  Go Back",
                    layout=widgets.Layout(width="98%", height="34px", border=f"1px solid {C_BORDER}"),
                    style=widgets.ButtonStyle(button_color=C_BACK_BG, font_color=C_BACK_TXT)
                )
                btn_up.on_click(lambda b: draw_browser(os.path.dirname(current_path)))
                buttons.append(btn_up)

            # ── list directory ───────────────────────────────────────────────
            try:
                items = os.listdir(current_path)
            except Exception:
                items = []

            folders = sorted([f for f in items if os.path.isdir(os.path.join(current_path, f))])

            # ── "Select Files" mode also needs valid files ───────────────────
            if upload_option == "Select Files (Drive)":
                valid_files = sorted([
                    f for f in items
                    if os.path.isfile(os.path.join(current_path, f)) and (is_image(f) or is_archive(f))
                ])
            else:
                valid_files = []   # ← directory mode: never show files

            MAX_ITEMS = 200

            # ── FOLDERS ──────────────────────────────────────────────────────
            for f in folders[:MAX_ITEMS]:
                fpath = os.path.join(current_path, f)

                if upload_option == "Select Directory (Drive)":
                    is_sel = fpath in selected_items

                    btn_open = widgets.Button(
                        description="Open",
                        layout=widgets.Layout(width="72px", height="32px", border=f"1px solid {C_BORDER}"),
                        style=widgets.ButtonStyle(button_color=C_OPEN, font_color=C_OPEN_TXT)
                    )
                    btn_open.on_click(lambda b, p=fpath: draw_browser(p))

                    btn_sel = widgets.Button(
                        description=f"  {f}",
                        icon="check" if is_sel else "folder",
                        layout=widgets.Layout(flex="1", height="32px", border=f"1px solid {C_BORDER}"),
                        style=widgets.ButtonStyle(
                            button_color=C_SEL if is_sel else C_FOLDER,
                            font_color=C_SEL_TXT if is_sel else C_FOLDER_TXT
                        )
                    )

                    def on_f_sel(b, p=fpath):
                        if p in selected_items:
                            selected_items.remove(p)
                            b.style.button_color = C_FOLDER
                            b.style.font_color   = C_FOLDER_TXT
                            b.icon = "folder"
                        else:
                            selected_items.add(p)
                            b.style.button_color = C_SEL
                            b.style.font_color   = C_SEL_TXT
                            b.icon = "check"

                    btn_sel.on_click(on_f_sel)
                    row = widgets.HBox(
                        [btn_open, btn_sel],
                        layout=widgets.Layout(width="98%", margin="1px 0")
                    )
                    buttons.append(row)

                else:
                    # "Select Files" mode — folders only navigate
                    btn_f = widgets.Button(
                        description=f"  {f}",
                        icon="folder",
                        layout=widgets.Layout(width="98%", height="32px", border=f"1px solid {C_BORDER}"),
                        style=widgets.ButtonStyle(button_color=C_FOLDER, font_color=C_FOLDER_TXT)
                    )
                    btn_f.on_click(lambda b, p=fpath: draw_browser(p))
                    buttons.append(btn_f)

            # ── FILES (only in "Select Files" mode) ──────────────────────────
            for f in valid_files[:MAX_ITEMS]:
                fpath = os.path.join(current_path, f)
                is_sel = fpath in selected_items
                ico = "archive" if is_archive(f) else "image"

                btn_file = widgets.Button(
                    description=f"  {f}",
                    icon="check" if is_sel else ico,
                    layout=widgets.Layout(width="98%", height="32px", border=f"1px solid {C_BORDER}"),
                    style=widgets.ButtonStyle(
                        button_color=C_SEL if is_sel else C_FOLDER,
                        font_color=C_SEL_TXT if is_sel else C_FOLDER_TXT
                    )
                )

                def on_file_click(b, p=fpath):
                    if p in selected_items:
                        selected_items.remove(p)
                        b.style.button_color = C_FOLDER
                        b.style.font_color   = C_FOLDER_TXT
                        b.icon = "archive" if is_archive(os.path.basename(p)) else "image"
                    else:
                        selected_items.add(p)
                        b.style.button_color = C_SEL
                        b.style.font_color   = C_SEL_TXT
                        b.icon = "check"

                btn_file.on_click(on_file_click)
                buttons.append(btn_file)

            # ── scroll box + confirm ─────────────────────────────────────────
            scroll_box = widgets.VBox(
                buttons,
                layout=widgets.Layout(
                    max_height="400px", overflow="auto",
                    padding="4px", width="99%",
                    background=C_BG,
                    border=f"1px solid {C_BORDER}",
                    border_radius="6px"
                )
            )

            btn_confirm = widgets.Button(
                description="✓  CONFIRM SELECTION",
                layout=widgets.Layout(
                    width="98%", height="40px",
                    margin="8px 0 0 0",
                    border=f"1px solid {C_BORDER}"
                ),
                style=widgets.ButtonStyle(
                    button_color=C_CONFIRM_BG,
                    font_color=C_CONFIRM_TXT,
                    font_weight="bold"
                )
            )

            def on_confirm(b):
                with result_out:
                    clear_output(wait=True)
                    if not selected_items:
                        print("❌ Select at least one item.")
                        return
                    print("⏳ Processing...")
                    clean_work_dir()
                    INPUT_FILES.clear()

                    if upload_option == "Select Directory (Drive)":
                        for fp in selected_items:
                            try:
                                for fname in os.listdir(fp):
                                    p = os.path.join(fp, fname)
                                    if os.path.isfile(p) and (is_image(fname) or is_archive(fname)):
                                        process_file(p)
                            except Exception as e:
                                print(f"❌ Could not process {fp}: {e}")
                    else:
                        for fp in selected_items:
                            process_file(fp)

                    show_results()

            btn_confirm.on_click(on_confirm)
            display(scroll_box, btn_confirm)

    display(main_out, result_out)
    draw_browser(ROOT_DRIVE)

In [ ]:
#@title ##**Run** { display-mode: "form" }
import subprocess, os, time
from IPython.display import display, clear_output, HTML
import ipywidgets as widgets

if 'INPUT_FILES' not in dir() or not INPUT_FILES:
    raise RuntimeError("❌ No images found. Run Cell 2 first.")
if not os.path.exists("/usr/local/bin/GeminiWatermarkTool"):
    raise RuntimeError("❌ Tool not installed. Run Cell 1 first.")

OUTPUT_DIR = "/content/processing_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

OUTPUT_FILES = []

total = len(INPUT_FILES)
print(f"⚙️  Processing {total} image(s)...\n")

progress_bar = widgets.IntProgress(
    value=0, min=0, max=total,
    description="0 / " + str(total),
    bar_style="info",
    style={"description_width": "80px", "bar_color": "#3A7D44"},
    layout=widgets.Layout(width="98%", height="28px")
)
display(progress_bar)

errors = []

for idx, input_path in enumerate(INPUT_FILES, start=1):
    fname = os.path.basename(input_path)
    out_name = os.path.splitext(fname)[0] + "_processed" + os.path.splitext(fname)[1]
    output_path = os.path.join(OUTPUT_DIR, out_name)

    r = subprocess.run(
        [
            "/usr/local/bin/GeminiWatermarkTool",
            "-i", input_path,
            "-o", output_path,
            "--denoise", "ns",
            "--force",
            "--no-banner",
        ],
        capture_output=True, text=True
    )

    if os.path.exists(output_path) and os.path.getsize(output_path) > 0:
        OUTPUT_FILES.append(output_path)
    else:
        errors.append(fname)

    progress_bar.value = idx
    progress_bar.description = f"{idx} / {total}"

if errors:
    print(f"\n⚠️  Failed ({len(errors)}): {', '.join(errors)}")

if OUTPUT_FILES:
    print(f"\n✅ Done — {len(OUTPUT_FILES)} of {total} processed successfully.")
    print("▶  Proceed to Cell 4 to download results.")
else:
    print("\n❌ No output files were created. Check your input images.")

In [ ]:
#@title ##**Download Results** { display-mode: "form" }
import os, shutil, zipfile
from IPython.display import display
from google.colab import files, drive
import ipywidgets as widgets

download_option = "Download to PC" #@param ["Download to PC", "Save to Google Drive"]

if 'OUTPUT_FILES' not in dir() or not OUTPUT_FILES:
    raise RuntimeError("❌ No processed files found. Run Cell 3 first.")

# ── helpers ───────────────────────────────────────────────────────────────────

def make_archive(output_files, archive_path="/content/results.zip"):
    with zipfile.ZipFile(archive_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for fp in output_files:
            zf.write(fp, os.path.basename(fp))
    return archive_path

# ── Download to PC ────────────────────────────────────────────────────────────

if download_option == "Download to PC":
    if len(OUTPUT_FILES) == 1:
        print(f"💾 Downloading: {os.path.basename(OUTPUT_FILES[0])}")
        files.download(OUTPUT_FILES[0])
        print("✅ Done.")
    else:
        archive = make_archive(OUTPUT_FILES)
        print(f"📦 Packing {len(OUTPUT_FILES)} files into results.zip...")
        files.download(archive)
        print("✅ Archive downloaded.")

# ── Save to Google Drive ──────────────────────────────────────────────────────

else:
    drive.mount('/content/drive')
    DRIVE_SAVE_DIR = "/content/drive/MyDrive"

    if len(OUTPUT_FILES) == 1:
        dest = os.path.join(DRIVE_SAVE_DIR, os.path.basename(OUTPUT_FILES[0]))
        shutil.copy(OUTPUT_FILES[0], dest)
        print(f"✅ Saved to Drive: {dest}")
    else:
        archive = make_archive(OUTPUT_FILES)
        dest = os.path.join(DRIVE_SAVE_DIR, "results.zip")
        shutil.copy(archive, dest)
        print(f"✅ Archive saved to Drive: {dest}")